# Managing Singleton Data with SingletonTable

## Introduction

This tutorial demonstrates how to use ``BaseSingletonTableSchema`` and ``SingletonTableManifestation`` to create single-row tables, commonly used for configuration settings.

### Why use a Singleton Table?

A **Singleton Table** is designed to hold exactly one row of data. Unlike standard tables where you manage multiple rows with unique identifiers, a Singleton Table guarantees that only one row exists. This is ideal for global application settings or configuration states where having multiple conflicting entries would be invalid.

The ``SingletonTableManifestation`` simplifies interaction by providing ``get_item()`` and ``set_item()`` methods, abstracting away the need to manage primary keys or check if the row already exists.

This tutorial will guide you through:
- Defining a Singleton Schema Mixin
- Creating a Singleton Manifestation
- Setting up the Database
- Managing the single item (Create, Get, Set)
- Handling Asynchronous operations

**Prerequisites:**
- Basic familiarity with Python and SQLAlchemy
- Installed package: ``sqlalchemyobjects``

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Usage](#Usage)
- [Async Usage](#Async-Usage)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)


## Importing the Module

We start by importing the necessary classes.

In [ ]:
from pathlib import Path
from sqlalchemy.orm import Mapped, mapped_column, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseSingletonTableSchema, SingletonTableManifestation

## Core Functionality

### 1. Define the Schema Mixin

We inherit from ``BaseSingletonTableSchema``.

In [ ]:
class ConfigTableSchema(BaseSingletonTableSchema):
    """Schema definition for configuration settings."""
    app_name: Mapped[str]
    debug_mode: Mapped[bool] = mapped_column(default=False)

### 2. Define the Manifestation

The manifestation provides specialized methods like ``get_item()`` and ``set_item()``.

In [ ]:
class ConfigTableManifestation(SingletonTableManifestation):
    """Manifestation class for Singleton configuration table."""


### 3. Define the Database Schema

Combine the TableSchema with the Declarative Base.

In [ ]:
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base class for the database schema."""

class ConfigTable(ConfigTableSchema, DatabaseSchema):
    """SQLAlchemy table definition for Configuration."""
    __tablename__ = "configuration"

### 4. Define the Database Class

Register the singleton table in the ``table_map``.

In [ ]:
class MyConfigDatabase(Database):
    """Database class managing the Configuration table."""
    schema = DatabaseSchema
    table_map = {
        "config": (ConfigTableManifestation, ConfigTable, {})
    }

    @property
    def config(self) -> ConfigTableManifestation:
        return self.tables["config"]

### 5. Setup Database

Initialize the database and access the manifestation.

In [ ]:
db_path = Path("tutorial_config.sqlite")
if db_path.exists():
    db_path.unlink()

database = MyConfigDatabase(path=db_path)
database.create_database()

# Access the table
config_table = database.config

## Usage

Singleton tables treat the single row as "the item".

In [ ]:
# Create the initial item
# You pass a dictionary of values.
print("Creating config...")
config_table.create_item({"app_name": "MyTutorialApp", "debug_mode": True})

# Get the item
current_config = config_table.get_item()
print(f"Current Config: {current_config.app_name}, Debug: {current_config.debug_mode}")

# Update (Set) the item
# This updates the existing single row.
print("Updating config...")
config_table.set_item({"debug_mode": False})

updated_config = config_table.get_item()
print(f"Updated Config: {updated_config.app_name}, Debug: {updated_config.debug_mode}")

## Async Usage

Like standard tables, singleton tables support async operations.

In [ ]:
import anyio

async def async_config_demo():
    """Demonstration of using the singleton table in asynchronous mode."""
    async_path = anyio.Path("tutorial_config_async.sqlite")
    if await async_path.exists():
        await async_path.unlink()

    # Use the same MyConfigDatabase class with async_engine=True
    async_db = MyConfigDatabase(path=str(async_path), async_engine=True)
    await async_db.create_database_async()

    # Access via property
    config_async = async_db.config

    # Create Async
    await config_async.create_item_async({"app_name": "AsyncApp", "debug_mode": True})

    # Get Async
    item = await config_async.get_item_async()
    print(f"Async Config: {item.app_name}")

    # Set Async
    await config_async.set_item_async({"app_name": "AsyncAppUpdated"})
    item = await config_async.get_item_async()
    print(f"Updated Async Config: {item.app_name}")

    await async_db.close_async()
    await async_path.unlink()

# Run async demo
await async_config_demo()

In [ ]:
# Cleanup
database.close()
if db_path.exists():
    db_path.unlink()

## API Highlights

- **``BaseSingletonTableSchema``**: TableSchema for single-row tables.
- **``SingletonTableManifestation``**: Interface with ``get_item``/``set_item``.
- **``MyConfigDatabase``**: Orchestrator mapping the singleton table.

## Troubleshooting / FAQs

- **Problem**: Calling ``get_item()`` returns ``None``.
  - **Solution**: The item must be created first using ``create_item()``.

## Conclusion and Next Steps

Singleton tables provide a clean interface for global configuration.

- **Reference**: See ``docs/concepts/comprehensive.rst``.